In [4]:
import numpy as np
import pandas as pd
import time
import sys
import os
from joblib import Parallel, delayed

sys.path.append("..")

ruta_data = "../data"
ruta_resultados = "../resultados"
archivo_csv = f"{ruta_resultados}/resultados_tests.csv"

# escenarios_a_actualizar = [7.1]
escenarios_a_actualizar = [i for i in range(1, 14) if i != 7] + [7.1, 7.2]
print("los escenarios a actualizar: ", escenarios_a_actualizar)

def procesar_muestra(escenario, i, x, y):
    sys.path.append("..")
    import metodos_estadisticos as tests

    resultados_locales = []
    metodos = [
    ("t_student",      lambda: tests.test_t_student(x, y)),
    ("welch",          lambda: tests.test_welch(x, y)),
    ("wilcoxon",       lambda: tests.test_wilcoxon(x, y)),
    ("ks_test",        lambda: tests.test_ks(x, y)),
    ("hodges_lehmann", lambda: tests.test_hodges_lehmann(x, y)),
    ("perm_media",     lambda: tests.test_permutacion(x, y, "media")),
    ("perm_mediana",   lambda: tests.test_permutacion(x, y, "mediana")),
    ("perm_trim",      lambda: tests.test_permutacion(x, y, "trim")),
    ]

    for nombre, fn in metodos:
        t0 = time.time()
        res = int(fn())
        resultados_locales.append({
            "escenario": escenario,
            "muestra": i,
            "metodo": nombre,
            "resultado": res,
            "tiempo": time.time() - t0
        })

    return resultados_locales

tareas = []
for esc in escenarios_a_actualizar:
    data = np.load(f"{ruta_data}/datos_escenario_{esc}.npz")
    for i, (x, y) in enumerate(zip(data["x"], data["y"])):
        tareas.append((esc, i, x, y))
print(len(tareas), "tareas a procesar")
resultados_anidados = Parallel(n_jobs=-1, verbose=10)(
    delayed(procesar_muestra)(esc, i, x, y) for esc, i, x, y in tareas
)

df_nuevos = pd.DataFrame([item for sublista in resultados_anidados for item in sublista])

if os.path.exists(archivo_csv):
    df_existente = pd.read_csv(archivo_csv)
    df_existente = df_existente[~df_existente["escenario"].isin(escenarios_a_actualizar)]
    df_final = pd.concat([df_existente, df_nuevos], ignore_index=True)
else:
    df_final = df_nuevos

df_final.to_csv(archivo_csv, index=False)

los escenarios a actualizar:  [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 7.1, 7.2]
7000 tareas a procesar


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    2.4s
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done  48 tasks      | elapsed:    2.9s
[Parallel(n_jobs=-1)]: Done  61 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done  74 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done  89 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 104 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done 121 tasks      | elapsed:    4.6s
[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:    5.0s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    5.7s
[Parallel(n_jobs=-1)]: Done 197 tasks      | elapsed:  